# Automated SAM2 reliability checks (not accuracy)

This notebook tests integrity, temporal consistency, prompt-box stability, and color-threshold sensitivity without human labels. Agreement and stability are reliability evidence only; they do not measure ground-truth accuracy.

In [1]:
from pathlib import Path
import json, os
import numpy as np
import pandas as pd
from IPython.display import display

from vial_analysis import AnalysisConfig, analyze_video, load_predictor
from vial_automated_validation import compare_runs, dataset_integrity_report, quality_report

ROOT = Path(os.getenv('VIAL_ROOT', '/home/hackathon5/vial_checks'))
OUTPUT_DIR = Path(os.getenv('VIAL_AUTOMATED_VALIDATION_DIR', str(ROOT / 'automated_validation')))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
COMBINED_PATH = ROOT / 'batch_results' / 'combined_measurements.csv'
VIDEO_DIR = ROOT / 'drive-download-20261031T172038Z-1-001'
combined = pd.read_csv(COMBINED_PATH)
print('Evidence class: AUTOMATED RELIABILITY ONLY — NOT ACCURACY')
print('Output directory:', OUTPUT_DIR)

Evidence class: AUTOMATED RELIABILITY ONLY — NOT ACCURACY
Output directory: /home/hackathon5/vial_checks/automated_validation


## Dataset integrity and quality signals

These checks can falsify broken output structure or unstable signals, but passing them does not prove correctness.

In [2]:
integrity = dataset_integrity_report(combined, expected_videos=26)
quality = quality_report(combined)
(OUTPUT_DIR / 'integrity_report.json').write_text(json.dumps(integrity, indent=2))
(OUTPUT_DIR / 'quality_report.json').write_text(json.dumps(quality, indent=2))
print('INTEGRITY:', integrity['integrity_label'])
print(quality['evidence_label'])
display(pd.Series(integrity, name='value').to_frame())
display(pd.Series(quality, name='value').to_frame())

INTEGRITY: PASS
AUTOMATED RELIABILITY ONLY — NOT ACCURACY


,value
integrity_label,PASS
video_count,26
row_count,11935
expected_video_count,26
unique_video_frame_keys,True
frame_indices_strictly_increasing,True
source_frames_strictly_increasing,True
timestamps_strictly_increasing,True
finite_timestamps,True
no_infinite_numeric_outputs,True


,value
evidence_label,AUTOMATED RELIABILITY ONLY — NOT ACCURACY
row_count,11935
measurement_rows,11200
measurement_coverage,0.938416
relative_change_rows,8066
no_change_rows,3134
no_vial_rows,735
large_shift_flag_rows,1265
downward_jump_flag_rows,162


## Real-video perturbation checks

When `VIAL_RUN_ROBUSTNESS=1`, the notebook reruns one stable-control video and one transition video with the prompt box shifted five pixels left/right and the Lab difference threshold changed from 6 to 4/8. It compares each run with the saved default output using status agreement, overlapping measurements, signal difference, and correlation.

In [3]:
comparison_columns = [
    'comparison_label', 'candidate_name', 'video_name', 'variant', 'vial_box',
    'de_threshold', 'row_count', 'status_agreement', 'measurement_overlap_rows',
    'measurement_overlap_rate', 'signal_mae_px',
    'signal_median_absolute_difference_px', 'signal_pearson_r',
]
RUN_ROBUSTNESS = os.getenv('VIAL_RUN_ROBUSTNESS', '0') == '1'
comparisons = []
if RUN_ROBUSTNESS:
    selected_videos = [
        'video_vial_fill_team-msoe-20261030T142247Z_20261030_143153_836260.mp4',
        'video_vial_fill_team-msoe-20261030T142247Z_20261030_144644_678654.mp4',
    ]
    variants = [
        ('repeat_default', (300, 60, 360, 420), 6.0),
        ('box_left_5px', (295, 60, 355, 420), 6.0),
        ('box_right_5px', (305, 60, 365, 420), 6.0),
        ('threshold_low_4', (300, 60, 360, 420), 4.0),
        ('threshold_high_8', (300, 60, 360, 420), 8.0),
    ]
    predictor = load_predictor('facebook/sam2.1-hiera-small')
    for video_name in selected_videos:
        reference = combined[combined.video_name == video_name].copy()
        for variant, vial_box, threshold in variants:
            config = AnalysisConfig(
                video_dir=VIDEO_DIR, work_dir=ROOT / 'batch_work',
                results_dir=OUTPUT_DIR / 'perturbation_results' / variant,
                stride=3, vial_box=vial_box, reference_frame_count=15,
                minimum_reference_frames=3, de_threshold=threshold, resume=False,
            )
            candidate, _ = analyze_video(VIDEO_DIR / video_name, config, predictor)
            report = compare_runs(reference, candidate, candidate_name=f'{video_name}:{variant}')
            report.update({
                'video_name': video_name, 'variant': variant,
                'vial_box': str(vial_box), 'de_threshold': threshold,
            })
            comparisons.append(report)
    comparison_frame = pd.DataFrame(comparisons)[comparison_columns]
    comparison_frame.to_csv(OUTPUT_DIR / 'robustness_comparisons.csv', index=False)
    print(f'Completed {len(comparison_frame)} real-video perturbation comparisons.')
    display(comparison_frame)
else:
    comparison_path = OUTPUT_DIR / 'robustness_comparisons.csv'
    if comparison_path.exists():
        comparison_frame = pd.read_csv(comparison_path)
        print(f'Robustness runs disabled; preserved {len(comparison_frame)} saved comparisons.')
    else:
        comparison_frame = pd.DataFrame(columns=comparison_columns)
        comparison_frame.to_csv(comparison_path, index=False)
        print('Robustness runs disabled; no saved comparisons were present.')

Robustness runs disabled; preserved 10 saved comparisons.


## Interpretation boundary

High agreement under perturbation supports repeatability and robustness. Low agreement identifies sensitivity that needs investigation. Neither result is accuracy: independent human labels remain necessary for fill-height error, no-vial classification accuracy, and SAM2 mask IoU or Dice.